In [ ]:
NAME = "" # put your full name here
COLLABORATORS = [] # list names of anyone you worked with on this homework.

# [ER 131] Homework 6: Gradient Descent

This homework focuses on gradient descent. By the end of this homework, you'll be able to implement gradient descent to fit simple non-linear models

### Table of Contents
* [Project](#project)<br>
1. [A Simple Model](#model)<br>
1. [Fitting the Model](#fitting)<br>
1. [Increasing Model Complexity](#complexity)<br>
1. [Gradient Descent](#gd)<br>

---

## Section A: Project (5 pts)<a id='project'></a>

**Question A.1 (1 pt)** Who will you be working with on the project? Enter first and last names of all your group members below. You should aim for groups of four. 

*YOUR ANSWER HERE*

**Question A.2 (3 pts)** List three (ideally related) prediction problems your group is interested in exploring. Your answer can be preliminary, and it's ok for your interests to evolve over the next few weeks. Make it clear that you're posing prediction problems, not inference problems. 

*YOUR ANSWER HERE*

**Question A.3 (2 pts)** Identify at least two datasets you could use to address one or more of these prediction problems. Give a one sentence description of each dataset, and provide the appropriate URL if available. 

*YOUR ANSWER HERE*

**Question A.4 (1 pt)** In a few sentences, explain the *motivation* behind your prediction problems. Who would be interested in seeing the results your prediction model? Why is it important to answer these questions? In other words, what resource allocation problems might your predictions help to address?

*YOUR ANSWER HERE*

---

## Introduction to Gradient Descent

Before we dive into the data and the homework, let's set up our motivation for exploring gradient descent. So far, we've been finding model parameters for linear regression by defining a loss function (a function that we want to minimize). Specifically, this loss function has been the mean squared error (MSE). The linear regression fitting that we did in homework 5 and lab 5 worked by solving for the parameter values that minimize the mean squared error on the training data. To minimize the MSE, we have to take its derivative, set it to zero, and solve for the parameters. In linear regression, we can solve this optimization problem analytically. 

However, this process isn't always feasible. When you have a model with a more complex form than linear regression, finding a derivative of the loss function and setting it to zero can be extremely difficult.  A second reason is that some of the loss functions you might encounter can't be massaged into a form that allows you to find the parameters algebraically. 

This is where gradient descent comes in. For complex models, it offers an easier way to find the minimum of the loss function. 

---
## Setup

In [ ]:
# import dependencies 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import csv
import warnings 
warnings.filterwarnings('ignore')
plt.style.use('fivethirtyeight') 

# Set some parameters
plt.rcParams['figure.figsize'] = (12, 9)
plt.rcParams['font.size'] = 14
np.set_printoptions(4)

In [ ]:
# We will use plot_3d helper function to help us visualize gradient
from hw06_utils import plot_3d

### Load Data
For this homework, we'll be working with simulated data.
Load the data.csv file into a pandas dataframe.  

In [ ]:
# Run this cell to load our sample data
data = pd.read_csv('data_hw06_2023.csv')
data.head()

In [ ]:
data.shape

---

## Section 1. A Simple Model<a id='model'></a>
Let's start by examining our data and creating a simple model that can represent this data.<br>

**Question 1.1 (2pts)** Define a function `scatter()` that produces a scatter plot. It should take as input the x and y values, and produce a scatter plot with generic "x" and "y" axis labels. Then, plot the $x$ and $y$ data from the `data` df you loaded above.<br><br>

In [ ]:
def scatter(x_var, y_var):
    """
    Objective: Generate a scatter plot using x_var (the independent variable) and y_var (the dependent variable)
    Inputs: 
        - x_var: the vector of values x
        - y_var: the vector of values y
    Output: A scatter plot
    """
    # create a scatter plot with x_var on the x-axis and y_var on the y-axis
    ... # YOUR CODE HERE
    plt.xlabel('x')
    plt.ylabel('y')
    return # You don't need to specify anything here; an "empty" return statement on a plot will allow us to add other features later.

In [ ]:
# # Replace ellipses with your code
# x = ... # YOUR CODE HERE
# y = ... # YOUR CODE HERE
# scatter(x,y)

**Question 1.2 (1pt):** Take a look at the distribution of the data. How can you describe the relationship between $x$ and $y$?

*YOUR ANSWER HERE*

**Question 1.3 (1pt):** For now, let's assume that the data follows a simple linear model, parametrized by $\theta$:

$\Large
\hat{y} = \theta \cdot x
$

Define a linear model function `linear_model()` that produces predicted $\hat{y}$ values (a vector) given $x$ (a vector) and $\theta$ (a scalar).

In [ ]:
def linear_model(x, theta):
    """
    Objective: predict y_hat values given x and theta
    Inputs:
        - x: the vector of values x
        - theta: the scalar theta
    Output: Returns the estimate of y given x and theta
    """
    ... # YOUR CODE HERE

In [ ]:
# run this cell, do not change it
assert linear_model(0, 1) == 0
assert linear_model(10, 10) == 100
assert np.sum(linear_model(np.array([3, 5]), 3)) == 24
assert linear_model(np.array([7, 8]), 4).mean() == 30

**Question 1.4 (1pt):** In class, we learned that the mean squared error (MSE) loss function is smooth and continuous with respect to the parameter. Let's use MSE loss to find an optimal value for $\theta$. First, define the MSE loss function `mse_loss` below, that calculates the value of MSE loss given a set of actual observations $y$ and predictions $\hat{y}$. Refer to previous labs or homeworks if you do not remember the equation for MSE. 

In [ ]:
def mse_loss(y, y_hat):
    """
    Objective: calculate the value of MSE loss given y and y_hat
    Inputs:
        - y: the vector of true values y
        - y_hat: the vector of predicted values y_hat
    Outputs: Returns the mean squared error (MSE) given y and y_hat.
    """
    ... # YOUR CODE HERE


In [ ]:
# run this cell, do not change it
assert mse_loss(2, 1) == 1
assert mse_loss(2, 0) == 4 
assert mse_loss(5, 1) == 16
assert mse_loss(np.array([5, 6]), np.array([1, 1])) == 20.5
assert mse_loss(np.array([1, 1, 1]), np.array([4, 1, 4])) == 6.0

**Question 1.5 (2pts):** Write a function `mse_plot()` that produces a line plot of MSE loss as a function of the coefficient $\theta$. Your function should take inputs $x$ and $y$, which are vectors of $x$ and $y$ observations, and input `thetas`, which is a list of possible thetas to test. You should end up with a plot of $\theta$ values on the x-axis, and the MSE loss corresponding with those $\theta$ values on the y-axis.  Make sure to label your axes and add a title. Use the functions you wrote for linear_model and mse_loss.
<br> 
<br>

In [ ]:
def mse_plot(x, y, thetas):
    """
    Objective: Plots the average MSE loss for given x and y as a function of theta.
    Inputs:
        - x: the vector of values x
        - y: the vector of values y
        - thetas: the vector containing different estimates of theta
    Outputs: line plot of MSE loss as a function of the coefficient theta
    """
    # Calculate the loss here for each value of theta
    ... # YOUR CODE HERE
    
    # Create your line plot 
    ... # YOUR CODE HERE
    plt.xlabel(...)
    plt.ylabel(...) 
    plt.title(...)
    return # Again, you can leave the return statement empty

**Question 1.6 (2pts):** Run the function `mse_plot()` using the $x$ and $y$ values from dataframe `data` above and a list of `thetas` (you can define this range yourself - try using the `np.linspace()` method, specifying a minimum and maximum value and the number of observations between these two values). 

What appears to be the optimal $\theta$ value based on the visualization? We'll call this value $\theta^*$.  Set the variable `theta_star_guess` to the value of $\theta$ that appears to minimize our loss based on the graph.

In [ ]:
# thetas = ...  # define a list of theta values to test. 
# mse_plot(x, y, thetas)
# theta_star_guess = ... # Your guess here

In [ ]:
assert mse_loss(3, 2) == 1
assert mse_loss(0, 10) == 100
assert 2 <= theta_star_guess <= 4

---
## Section 2: Fitting our Simple Model<a id='fitting'></a>
Now that we have defined a simple linear model and loss function, let's begin working on fitting our model to the data.

**Question 2.1 (1pt):** Let's confirm our visual findings for our optimal coefficient $\theta^*$. First, identify the analytical solution for the optimal $\theta^*$ that minimizes average MSE loss. Of these three options, which correctly gives the formula that tells us what $\theta^*$ is given $n$ observations of $x$ and $y$? 

1. $\Large {\theta}^* = \frac{\sum_n x_iy_i}{\sum_n x_i}$

2. $\Large {\theta}^* = \frac{\sum_n x_i + y_i}{\sum_n x_i^2}$

3. $\Large {\theta}^* = \frac{\sum_n x_iy_i}{\sum_n x_i^2}$

*YOUR ANSWER HERE*

**Question 2.2 (1pt):** 
Use the analytic solution for $\theta^*$ to implement the function `find_theta`, which calculates the numerical value of $\theta^*$ based on our data $x$, $y$. 

In [ ]:
def find_theta(x, y):
    """
    Objective: Find optimal theta given x and y
    Inputs:
        - x: the vector of values x
        - y: the vector of values y
    Output: the optimal theta_star 
    """
    # YOUR CODE HERE
    return ...

t_star = find_theta(x, y) # Your code here to get theta star

In [ ]:
# run this cell; do not change it
print(f'theta_opt = {t_star}')
assert 3 <= t_star <= 4

**Question 2.3 (1pt):** Now, let's plot our loss function again using the `mse_plot()` function. This time, add a vertical line at the optimal value of theta (i.e. plot the line $x = \theta^*$). The function `plt.axvline()` is helpful here.

In [ ]:
# mse_plot(...) # plot the loss function
# ... # add a vertical line

**Question 2.4 (1pt):** We now have an optimal value for $\theta$ that minimizes our loss. In the cell below, plot the scatter plot of the data from Question 1.1 (you can reuse the `scatter()` function here). Add the best-fit line $\hat{y} = \theta^* \cdot x$ using the $\theta^*$ you computed above.

In [ ]:
# YOUR CODE HERE
# Plot the observations as a scatter plot
# add a line of best fit based on your calculated value of t_star

**Question 2.5 (1pt):** Great! It looks like our estimate for $\theta$ is able to capture a lot of the data with a single parameter. Now let's try to plot the residual to see what we've missed.<br>  

The residual is defined as $r=y-\theta^* \cdot x$. Below, write a function to find the residual and plot the residuals as a function of the independent variable in a scatter plot (use `plt.axhline()`). Plot a horizontal line at $y=0$ to assist with visualization. Add axis labels.

In [ ]:
def visualize_residual(x, y):
    """
    Objective: Visualize the residuals against a horizontal line at y = 0
    Inputs:
        - x: the vector of values x
        - y: the vector of values y
    Outputs: Plot a scatter plot of the residuals, the remaining 
    values after removing the linear model from our data.
    """
    # calculate residual
    r = ...
    
    # plot residual, including axis labels and vertical line at y=0
    ... 

visualize_residual(x, y)
plt.show()

**Question 2.6 (1pt):** What does the residual look like? Do you notice a relationship between $x$ and $r$?

*YOUR ANSWER HERE*

---
## Section 3: Increasing Model Complexity<a id='complexity'></a>

It looks like the residuals follow a sinusoidal pattern. In other words, the trend in our original data seems to have a linear component and a sinusoidal component. To address this discovery, we'll propose a new model and find optimal parameters to fit the model to the data. Consider the following model:

$$\Large
\hat{y} = \theta_1x + cos(\theta_2x)
$$

Now, our model is parameterized by both $\theta_1$ and $\theta_2$ (or, composed together, by the vector $\vec{\theta}$).

Note that a generalized cosine function $a\cos(bx+c)$ has three parameters: amplitude scaling parameter $a$, frequency parameter $b$ and phase shifting parameter $c$. For the sake of this assignment, you can assume that the scaling and shifting parameter ($a$ and $c$ in this case) are 1 and 0 respectively. 

**Question 3.1 (1pt):** As in Question 1.3, write a function `cos_model` that predicts a value $\hat{y}$ given inputs $x$, $\theta_1$, and $\theta_2$ based on our new model. Hint: you may find the `np.cos` function helpful.

In [ ]:
def cos_model(x, theta_1, theta_2):
    """
    Objective: Predict the estimate of y given x, theta_1, theta_2
    Inputs:
        - x: the vector of values x
        - theta_1: the scalar value theta_1
        - theta_2: the scalar value theta_2
    Outputs: predicted value for y_hat
    """
    # YOUR CODE HERE
    y_hat = ...
    return y_hat

In [ ]:
print(np.isclose(cos_model(1, 1, np.pi), 0))
# Check that we accept x as arrays
assert len(cos_model(x, 2, 2)) > 1

**Question 3.2 (1pt):** In this question your job is to match the left and right sides of the equations for:
1. The MSE loss for our model, $\hat{y} = \theta_1x + cos(\theta_2x)$.  We'll call that $L(x, y, \theta_1, \theta_2)$.
2. The partial derivatives of the our model's loss functions, $\frac{\partial L }{\partial \theta_1} and \frac{\partial L }{\partial \theta_2}$. 

Notice that we now have $\vec{x}$ and $\vec{y}$ instead of $x$ and $y$. This means that when determining the loss function $L(x, y, \theta_1, \theta_2)$, you'll need to take the average of the squared losses for each $y_i$, $\hat{y_i}$ pair.

As your answer below, match each numbered item to the corresponding letter.

1. $L(x, y, \theta_1, \theta_2)$

2. $\frac{\partial L}{\partial \theta_1}$ 

3. $\frac{\partial L}{\partial \theta_2}$ 

A. $\frac{2}{n} \sum_{i=1}^n (x_i y_i \sin(\theta_2 x_i) - \theta_1 x_i ^ 2 \sin(\theta_2 x_i) - x_i \sin(\theta_2 x_i)\cos(\theta_2 x_i))$

B.$\frac{1}{n} \sum_{i=1}^n (y_i - \theta_1 x_i - \cos(\theta_2 x_i)) ^ 2$

C. $-\frac{2}{n} \sum_{i=1}^n (x_i y_i - \theta_1 x_i ^ 2 - x_i \cos(\theta_2 x_i))$


*Your answer*: <br>
1. ... <br>
2. ... <br>
3. ... <br>

**Question 3.3 (2pts):** Now, implement the functions `dt1` and `dt2`, which should compute $\frac{\partial L }{\partial \theta_1}$ and $\frac{\partial L }{\partial \theta_2}$ respectively. Use the formulas for $\frac{\partial L }{\partial \theta_1}$ and $\frac{\partial L }{\partial \theta_2}$ from the previous exercise. In the functions below, the parameter `theta` is a vector that looks like $[ \theta_1, \theta_2 ]$.

In [ ]:
def dt1(x, y, theta):
    """
    Objective: Compute the numerical value of the partial derivative of MSE loss with respect to theta_1
    Inputs:
        - x: the vector of all x values
        - y: the vector of all y values
        - theta: the vector of values theta
    Output: the numerical value of the partial derivative of MSE loss with respect to theta_1
    """
    # YOUR CODE HERE

In [ ]:
def dt2(x, y, theta):
    """
    Objective: Compute the numerical value of the partial derivative of MSE loss with respect to theta_2
    Inputs:
        - x: the vector of all x values
        - y: the vector of all y values
        - theta: the vector of values theta
    Output: the numerical value of the partial derivative of MSE loss with respect to theta_2
    """
    # YOUR CODE HERE

In [ ]:
# This function calls dt1 and dt2 and returns the gradient dt. It is already implemented for you.
def dt(x, y, theta):
    """
    Objective: Calculate the gradient of MSE loss with respect to vector theta
    Keyword arguments:
        - x: the vector of values x
        - y: the vector of values y
        - theta: the vector of values theta
    Output: the gradient dt
    """
    return np.array([
        dt1(x, y, theta),
        dt2(x, y, theta)
    ])

In [ ]:
# check your solution
assert np.isclose(dt1(x, y, [0, np.pi]), -217.22670090329441)
assert np.isclose(dt2(x, y, [0, np.pi]), 1.3275857635175532)

---
## Section 4: Gradient Descent<a id='gd'></a>
Now try to solve for the optimal $\theta^*$ analytically...

**Just kidding!**

You can try but we don't recommend it. When finding an analytic solution becomes difficult or impossible, we resort to alternative optimization methods for finding an approximate solution.

So let's try implementing a numerical optimization method: gradient descent!


**Question 4.1 (3pts):** Implement the `grad_desc` function that performs gradient descent for a finite number of iterations. This function takes in array $x$, array $y$, and an initial value for $\theta$ (`theta`). `alpha` will be the learning rate (or step size, whichever term you prefer). In this part, we'll use a static learning rate that is the same at every time step. 

At each time step, use the gradient and `alpha` to update your current `theta`. Also at each time step, be sure to save the current `theta` in `theta_history`, along with the MSE loss (computed with the current `theta`) in `loss_history`.

Hints:
- Write out the gradient update equation (1 step). What variables will you need for each gradient update? Of these variables, which ones do you already have, and which ones will you need to recompute at each time step?
- You may need a loop here to update `theta` several times.

In [ ]:
# Run this cell
def init_t():
    """Creates an initial theta [2, -2] as a starting point for gradient descent"""
    return np.array([2,-2])

In [ ]:
def grad_desc(x, y, theta, num_iter=20, alpha=0.0001):
    """
    Objective: Run gradient descent update for a finite number of iterations and static learning rate

    Inputs:
        - x: the vector of values x
        - y: the vector of values y
        - theta: the vector of values theta to use at first iteration
        - num_iter: the max number of iterations
        - alpha: the learning rate (also called the step size)
    
    Outputs:
        - theta: the optimal value of theta after num_iter of gradient descent
        - theta_history: the series of theta values over each iteration of gradient descent
        - loss_history: the series of loss values over each iteration of gradient descent
    """
    
    # YOUR CODE HERE
    
    
    return theta, theta_history, loss_history

Now, run the code below to use gradient descent to fit the model. 

In [ ]:
t = init_t() # set initial theta values
t_est, ts, loss = grad_desc(x, y, t)

In [ ]:
# check your output
assert len(ts) == len(loss) == 20 # theta history and loss history are 20 items in them
assert ts[0].shape == (2,) # theta history contains theta values
assert np.isscalar(loss[0]) # loss history is a list of scalar values, not vector
assert loss[1] - loss[-1] > 0 # loss is decreasing

**Question 4.2 (2pts):** Let's visually inspect the results. Plot the predicted y values over the original scatter plot. Include a legend on your plot. Did gradient descent successfully find the value of $\theta$ that minimizes MSE?

In [ ]:
# YOUR CODE HERE

*YOUR ANSWER HERE*

**Question 4.3 (1 pt)** Assume $y=\theta_1 x + cos(\theta_2 x)$ is the correct functional form. Why didn't gradient descent manage to find a model that fits the observed data better? (Hint: it might have something to to with the default parameters in our implementation of the gradient descent method). 

*YOUR ANSWER HERE*

**Question 4.4 (2pt):** Now let's visualize gradient descent to see how it converges. Plot a line plot with the loss values on the y-axis and the iteration number (i.e., 0-20) on the x-axis. What you you observe about the plot?

In [ ]:
# YOUR CODE HERE

*YOUR ANSWER HERE*

**Question 4.5 (2pts):** Create a single plot that shows the loss value (y-axis) versus the iteration (x-axis) for different values of `alpha`: try using `alpha` = 0.01, `alpha` = 0.001, and `alpha` = 0.0001. Add a legend. How does the loss value change over different iterations when alpha varies? Based on what you know about gradient descent, why does the loss value change in this way?<br>

In [ ]:
# YOUR CODE HERE

*YOUR ANSWER HERE*

**Question 4.6 (2 pts)** Using your results from the previous problem, pick the parameter values that minimize MSE. Using these parameter values, make predictions for every observed value of x in our dataset. Finally, plot the predicted values over the actual observations, as you did for Quesiton 4.2. How well does this model seem to fit the data?

In [ ]:
# YOUR CODE HERE

*YOUR ANSWER HERE*


----

## Bibliography

+ Data 100 - HW 5: Modeling, Estimation and Gradient Descent

<hr/>

Data Science Modules: http://data.berkeley.edu/education/modules